
<div  style="text-align: center; line-height: 0; padding-top: 9px;">
  <img src="https://raw.githubusercontent.com/derar-alhussein/Databricks-Certified-Data-Engineer-Associate/main/Includes/images/bookstore_schema.png" alt="Databricks Learning" style="width: 600">
</div>

## Querying JSON 

In [0]:
%python
def path_exists(path):
  try:
    dbutils.fs.ls(path)
    return True
  except Exception as e:
    msg = str(e)
    if ("com.databricks.sql.io.CloudFileNotFoundException" in msg
        or "java.io.FileNotFoundException" in msg):
      return False
    else:
      raise

In [0]:
%python
def download_dataset(source, target):
    files = dbutils.fs.ls(source)

    for f in files:
        source_path = f"{source}/{f.name}"
        target_path = f"{target}/{f.name}"
        if not path_exists(target_path):
            print(f"Copying {f.name} ...")
            dbutils.fs.cp(source_path, target_path, True)

In [0]:
CREATE CATALOG IF NOT EXISTS demo;
2
CREATE SCHEMA IF NOT EXISTS demo.bookstore;
3
CREATE VOLUME demo.bookstore.raw_data;

In [0]:
%python
data_source_uri = "s3://dalhussein-courses/datasets/bookstore/v1/"
dataset_bookstore = '/Volumes/dbx_catalog/dbx_schema/bookstore/raw/'
data_catalog = 'dbx_catalog'


In [0]:
%python
def get_index(dir):
    files = dbutils.fs.ls(dir)
    index = 0
    if files:
        file = max(files).name
        index = int(file.rsplit('.', maxsplit=1)[0])
    return index+1

In [0]:
%python

def set_current_catalog(catalog_name):
    spark.sql(f"USE CATALOG {catalog_name}")

In [0]:
%python
# Structured Streaming
streaming_dir = f"{dataset_bookstore}/orders-streaming"
raw_dir = f"{dataset_bookstore}/orders-raw"

def load_file(current_index):
    latest_file = f"{str(current_index).zfill(2)}.parquet"
    print(f"Loading {latest_file} file to the bookstore dataset")
    dbutils.fs.cp(f"{streaming_dir}/{latest_file}", f"{raw_dir}/{latest_file}")

    
def load_new_data(all=False):
    index = get_index(raw_dir)
    if index >= 10:
        print("No more data to load\n")

    elif all == True:
        while index <= 10:
            load_file(index)
            index += 1
    else:
        load_file(index)
        index += 1

In [0]:
%python
# DLT
streaming_orders_dir = f"{dataset_bookstore}/orders-json-streaming"
streaming_books_dir = f"{dataset_bookstore}/books-streaming"

raw_orders_dir = f"{dataset_bookstore}/orders-json-raw"
raw_books_dir = f"{dataset_bookstore}/books-cdc"

def load_json_file(current_index):
    latest_file = f"{str(current_index).zfill(2)}.json"
    print(f"Loading {latest_file} orders file to the bookstore dataset")
    dbutils.fs.cp(f"{streaming_orders_dir}/{latest_file}", f"{raw_orders_dir}/{latest_file}")
    print(f"Loading {latest_file} books file to the bookstore dataset")
    dbutils.fs.cp(f"{streaming_books_dir}/{latest_file}", f"{raw_books_dir}/{latest_file}")

    
def load_new_json_data(all=False):
    index = get_index(raw_orders_dir)
    if index >= 10:
        print("No more data to load\n")

    elif all == True:
        while index <= 10:
            load_json_file(index)
            index += 1
    else:
        load_json_file(index)
        index += 1

In [0]:
%python
download_dataset(data_source_uri, )

In [0]:
%python
print dataset_bookstore

In [0]:
%python
files = dbutils.fs.ls(f"{dataset_bookstore}/customers-json")
display(files)

In [0]:
SELECT * FROM json.`dbfs:/Volumes/dbx_catalog/dbx_schema/bookstore/raw/customers-json/export_001.json`

In [0]:
%python
dbutils.widgets.text(
    "dataset_bookstore",
    "dbfs:/Volumes/dbx_catalog/dbx_schema/bookstore/raw",
    "Dataset Path"
)

In [0]:
SELECT * FROM json.`${dataset_bookstore}/customers-json/export_*.json`

In [0]:
SELECT * FROM json.`/Volumes/dbx_catalog/dbx_schema/bookstore/raw/customers-json`

In [0]:
SELECT count(*) FROM json.`/Volumes/dbx_catalog/dbx_schema/bookstore/raw/customers-json`

In [0]:
 SELECT *,
    _metadata.file_path source_file
  FROM json.`/Volumes/dbx_catalog/dbx_schema/bookstore/raw/customers-json`;

## Querying text Format

In [0]:
SELECT * FROM text.`/Volumes/dbx_catalog/dbx_schema/bookstore/raw/customers-json`

## Querying binaryFile Format

In [0]:
SELECT * FROM binaryFile.`/Volumes/dbx_catalog/dbx_schema/bookstore/raw/customers-json`


## Querying CSV 

In [0]:
SELECT * FROM csv.`/Volumes/dbx_catalog/dbx_schema/bookstore/raw/books-csv`

In [0]:
CREATE TABLE books_csv
  (book_id STRING, title STRING, author STRING, category STRING, price DOUBLE)
USING CSV
OPTIONS (
  header = "true",
  delimiter = ";"
)
LOCATION "dbfs:/Volumes/dbx_catalog/dbx_schema/bookstore/raw/books-csv"

In [0]:
CREATE TABLE books_csv
  (book_id STRING, title STRING, author STRING, category STRING, price DOUBLE)

In [0]:
CREATE TEMP VIEW books_csv
(book_id STRING, title STRING, author STRING, category STRING, price DOUBLE)
using CSV
OPTIONS (
    path = "dbfs:/Volumes/dbx_catalog/dbx_schema/bookstore/raw/books-csv",
    header = "true",
    delimiter = ";"
)

In [0]:
SELECT * FROM books_csv


## Limitations of Non-Delta Tables

In [0]:
DESCRIBE detail books_csv

In [0]:
INSERT INTO books_csv
SELECT * FROM books_csv

In [0]:
%python
files = dbutils.fs.ls(f"{dataset_bookstore}/books-csv")
display(files)

In [0]:
%python
(spark.read
        .table("books_csv")
      .write
        .mode("append")
        .format("csv")
        .option('header', 'true')
        .option('delimiter', ';')
        .save(f"{dataset_bookstore}/books-csv"))

In [0]:
%python
files = dbutils.fs.ls(f"{dataset_bookstore}/books-csv")
display(files)

In [0]:
SELECT COUNT(*) FROM books_csv

In [0]:
REFRESH TABLE books_csv

In [0]:
SELECT COUNT(*) FROM books_csv

## CTAS Statements

In [0]:
CREATE TABLE customers AS
SELECT * FROM json.`${dataset.bookstore}/customers-json`;

DESCRIBE EXTENDED customers;

In [0]:
CREATE TABLE books_unparsed AS
SELECT * FROM csv.`${dataset.bookstore}/books-csv`;

SELECT * FROM books_unparsed;

In [0]:
CREATE TEMP VIEW books_tmp_vw
   (book_id STRING, title STRING, author STRING, category STRING, price DOUBLE)
USING CSV
OPTIONS (
  path = "${dataset.bookstore}/books-csv/export_*.csv",
  header = "true",
  delimiter = ";"
);

CREATE TABLE books AS
  SELECT * FROM books_tmp_vw;
  
SELECT * FROM books

In [0]:
DESCRIBE EXTENDED books

In [0]:
CREATE TABLE books
SELECT * FROM read_files("dbfs:/Volumes/dbx_catalog/dbx_schema/bookstore/raw/books-csv/",
format => 'csv',
delimiter =>';',
header =>'true'
);

In [0]:
select * from books